In [14]:
import pandas as pd
import sqlite3

df = pd.read_csv(
    "reseñas.csv",
    sep=";",
    encoding="utf-8"
)

df = df.dropna()
print(df.head())

# creo una base de datos (NUEVO archivo .db)
conexion = sqlite3.connect("reseñas.db")
# lo guardo como tabla 
df.to_sql("reseñas", conexion, if_exists="replace", index=False)

# INICIO DE LAS CONSULTAS
resultado = pd.read_sql("""
SELECT Restaurante, COUNT(*) as cantidad
FROM reseñas
GROUP BY Restaurante
""", conexion)

        Restaurante    Barrio            Tipo Precio  Rating  \
0  Orgánico Express  Recoleta          Vegano    $$$       4   
1      Fresh Market   Almagro  Café saludable      $       3   
2    Semilla Urbana   Almagro        Sin TACC     $$       1   
3    Semilla Urbana  Recoleta         Natural    $$$       2   
4       Vida Vegana   Palermo        Orgánico    $$$       5   

                            Reseña  
0  Ingredientes de primera calidad  
1                   Rico pero caro  
2       Las porciones son pequeñas  
3                No vale el precio  
4   Excelente calidad y muy fresco  


In [15]:
valores = pd.read_sql("""
SELECT Restaurante, COUNT(*) AS menciones_valoradas
FROM reseñas
WHERE Reseña LIKE '%saludable%'
   OR Reseña LIKE '%porciones%'
   OR Reseña LIKE '%deliciosa%'
   OR Reseña LIKE '%fresco%'
   OR Reseña LIKE '%calidad%'
GROUP BY Restaurante
ORDER BY menciones_valoradas DESC
""", conexion)
print(valores.head(10))

        Restaurante  menciones_valoradas
0    Semilla Urbana                   11
1  Orgánico Express                   10
2        Verde Raíz                    8
3    Raíces Urbanas                    8
4        Green Spot                    8
5      La Huerta BA                    7
6           EcoBite                    6
7       Vida Vegana                    5
8     Sabor Natural                    4
9      Fresh Market                    3


In [16]:
#Qué valoran más los consumidores?
valores_ej = pd.read_sql("""
SELECT DISTINCT Reseña, Restaurante
FROM reseñas
WHERE Reseña LIKE '%saludable%'
   OR Reseña LIKE '%ambiente%'
   OR Reseña LIKE '%lindo%'
   OR Reseña LIKE '%fresco%'
   OR Reseña LIKE '%atencion%'
ORDER BY RANDOM()
LIMIT 5;
""", conexion)
print(valores_ej)


                                    Reseña     Restaurante
0  Muy buena atención y ambiente agradable     Vida Vegana
1       Ambiente lindo pero servicio lento  Semilla Urbana
2             Comida deliciosa y saludable     Vida Vegana
3           Excelente calidad y muy fresco   Sabor Natural
4  Muy buena atención y ambiente agradable    Fresh Market


ANALISIS INICIAL:

Podemos ver que aspectos como el ambiente, atencion, comida saludable y la frescura de los alimentos
corresponden a la parte mas importante en lo que se fijan las personas.

In [17]:
problemas_ej = pd.read_sql("""
SELECT DISTINCT Reseña, Restaurante
FROM reseñas
WHERE Reseña LIKE '%atención lenta%'
   OR Reseña LIKE '%demora%'
   OR Reseña LIKE '%porciones%'
   OR Reseña LIKE '%mala%'
   OR Reseña LIKE '%desastroza%'
   OR Reseña LIKE '%horrenda%'
   OR Reseña LIKE '%espera%'
   OR Reseña LIKE '%no volver%'
ORDER BY RANDOM()
LIMIT 3;
""", conexion)

print(problemas_ej)


                             Reseña    Restaurante
0  Demasiada espera para los platos  Sabor Natural
1        Las porciones son pequeñas    Vida Vegana
2        Las porciones son pequeñas     Verde Raíz


ANALISIS DE LOS DISGUTOS MAS USUALES.
Si bien este flujo se repite en varios restaurantes, se puede ver que la mala atencion
el tamaño de las porciones y la espera entre los platos son causa principal de malas reseñas.

In [18]:
precio_ej = pd.read_sql("""
SELECT DISTINCT Reseña, Restaurante, Tipo
FROM reseñas
WHERE Tipo IN ('Vegano','Orgánico','Natural','Sin TACC')
  AND (Reseña LIKE '%caro%' OR Reseña LIKE '%precio%' OR Reseña LIKE '%vale%')
ORDER BY RANDOM()
LIMIT 4;
""", conexion)

print(precio_ej)


                         Reseña     Restaurante      Tipo
0             No vale el precio    La Huerta BA    Vegano
1  Muy caro para lo que ofrecen     Vida Vegana    Vegano
2  Muy caro para lo que ofrecen  Raíces Urbanas  Orgánico
3  Muy caro para lo que ofrecen     Vida Vegana  Sin TACC


ANALISIS DEL PRECIO COMO UN OBSTACULO
Segun lo que muestran los filtros, es comun que, aunque la calidad del producto sea la adecuada, o incluso supere
estandares. Si los precios son desproporcionados, estos SI suponen una barrera a la hora de elegir un restaurante de comida saludable